# Tourism Experience Analytics
## Classification, Rating Prediction and Recommendation System

### Project Objectives
- **Regression:** Predict attraction ratings.
- **Classification:** Predict user visit mode.
- **Recommendation:** Recommend tourist attractions based on ratings and preferences.

> Upload all tourism CSV files to Colab before running the project.


## 1. Install Required Libraries

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn joblib


## 2. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

RANDOM_STATE = 42


## 3. Upload Dataset Files

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print("-", filename)


## 4. Automatically Detect CSV Files

This cell lists every uploaded CSV file.  
After uploading, update the file mapping in the next cell if required.


In [ ]:
import glob
csv_files = glob.glob("*.csv")

print("CSV files found:")
for f in csv_files:
    print("-", f)


## 5. Configure Dataset Filenames

In [ ]:
# CHANGE THESE FILENAMES IF YOUR DATASET USES DIFFERENT NAMES

FILES = {
    "transaction": "Transaction.csv",
    "user": "User.csv",
    "city": "City.csv",
    "type": "Type.csv",
    "visit_mode": "VisitMode.csv",
    "continent": "Continent.csv",
    "country": "Country.csv",
    "region": "Region.csv",
    "item": "Item.csv"
}

print(FILES)


## 6. Load Datasets

In [ ]:
def load_csv_safely(filename):
    if os.path.exists(filename):
        return pd.read_csv(filename)
    print(f"WARNING: File not found -> {filename}")
    return None

datasets = {name: load_csv_safely(path) for name, path in FILES.items()}

for name, data in datasets.items():
    print("\n" + "="*60)
    print(name.upper())
    print("="*60)
    if data is not None:
        print("Shape:", data.shape)
        display(data.head())
        print("Columns:", list(data.columns))


## 7. Create the Consolidated Dataset

In [ ]:
transaction_df = datasets["transaction"]
user_df = datasets["user"]
city_df = datasets["city"]
type_df = datasets["type"]
continent_df = datasets["continent"]
country_df = datasets["country"]
region_df = datasets["region"]
item_df = datasets["item"]

if transaction_df is None or user_df is None or item_df is None:
    raise ValueError("Transaction, User and Item datasets are required.")

df = transaction_df.copy()

# User information
if "UserId" in df.columns and user_df is not None:
    df = df.merge(user_df, on="UserId", how="left", suffixes=("", "_user"))

# Attraction information
if "AttractionId" in df.columns and item_df is not None:
    df = df.merge(item_df, on="AttractionId", how="left", suffixes=("", "_item"))

# Attraction type information
if type_df is not None and "AttractionTypeId" in df.columns and "AttractionTypeId" in type_df.columns:
    df = df.merge(type_df, on="AttractionTypeId", how="left", suffixes=("", "_type"))

# City information
if city_df is not None and "CityId" in df.columns and "CityId" in city_df.columns:
    df = df.merge(city_df, on="CityId", how="left", suffixes=("", "_city"))

# Country information
if country_df is not None and "CountryId" in df.columns and "CountryId" in country_df.columns:
    df = df.merge(country_df, on="CountryId", how="left", suffixes=("", "_country"))

# Region information
if region_df is not None and "RegionId" in df.columns and "RegionId" in region_df.columns:
    df = df.merge(region_df, on="RegionId", how="left", suffixes=("", "_region"))

# Continent information
if continent_df is not None and "ContinentId" in df.columns and "ContinentId" in continent_df.columns:
    df = df.merge(continent_df, on="ContinentId", how="left", suffixes=("", "_continent"))

print("Final merged dataset shape:", df.shape)
display(df.head())
print("\nColumns:")
print(list(df.columns))


## 8. Data Cleaning

In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

# Fill numerical missing values
numeric_cols = df.select_dtypes(include=["number"]).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

# Remove duplicates
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Removed {before - after} duplicate rows.")
print("Final shape:", df.shape)


## 9. Exploratory Data Analysis (EDA)

In [ ]:
print(df.describe(include="all"))


In [ ]:
if "Rating" in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df["Rating"].dropna(), bins=20, kde=True)
    plt.title("Distribution of Attraction Ratings")
    plt.xlabel("Rating")
    plt.show()


In [ ]:
if "VisitMode" in df.columns:
    plt.figure(figsize=(10,5))
    order = df["VisitMode"].value_counts().index
    sns.countplot(data=df, x="VisitMode", order=order)
    plt.title("Visit Mode Distribution")
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
if "AttractionType" in df.columns:
    top_types = df["AttractionType"].value_counts().head(10)
    plt.figure(figsize=(12,6))
    sns.barplot(x=top_types.index, y=top_types.values)
    plt.title("Top 10 Attraction Types")
    plt.xlabel("Attraction Type")
    plt.ylabel("Number of Visits")
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
if "VisitMode" in df.columns and "Rating" in df.columns:
    plt.figure(figsize=(10,5))
    sns.barplot(data=df, x="VisitMode", y="Rating")
    plt.title("Average Rating by Visit Mode")
    plt.xticks(rotation=45)
    plt.show()


# PART 1 — Regression: Predict Attraction Rating

In [ ]:
# Select useful columns that actually exist in the dataset
candidate_features = [
    "VisitYear", "VisitMonth",
    "Continent", "Region", "Country",
    "CityName", "AttractionType"
]

regression_features = [c for c in candidate_features if c in df.columns]
print("Regression features:", regression_features)

if "Rating" not in df.columns:
    raise ValueError("Rating column was not found.")

X_reg = df[regression_features].copy()
y_reg = df["Rating"].copy()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

categorical_features_reg = X_reg.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features_reg = [c for c in X_reg.columns if c not in categorical_features_reg]

preprocessor_reg = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features_reg),
        ("numerical", "passthrough", numerical_features_reg)
    ],
    remainder="drop"
)

print("Train shape:", X_train_reg.shape)
print("Test shape:", X_test_reg.shape)


In [ ]:
# Linear Regression
linear_regression = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", LinearRegression())
])

linear_regression.fit(X_train_reg, y_train_reg)
linear_predictions = linear_regression.predict(X_test_reg)

linear_mae = mean_absolute_error(y_test_reg, linear_predictions)
linear_mse = mean_squared_error(y_test_reg, linear_predictions)
linear_r2 = r2_score(y_test_reg, linear_predictions)

print("Linear Regression Results")
print("MAE:", linear_mae)
print("MSE:", linear_mse)
print("R2 Score:", linear_r2)


In [ ]:
# Random Forest Regressor
rf_regressor = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_regressor.fit(X_train_reg, y_train_reg)
rf_predictions = rf_regressor.predict(X_test_reg)

rf_mae = mean_absolute_error(y_test_reg, rf_predictions)
rf_mse = mean_squared_error(y_test_reg, rf_predictions)
rf_r2 = r2_score(y_test_reg, rf_predictions)

print("Random Forest Regressor Results")
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("R2 Score:", rf_r2)


In [ ]:
regression_results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor"],
    "MAE": [linear_mae, rf_mae],
    "MSE": [linear_mse, rf_mse],
    "R2 Score": [linear_r2, rf_r2]
})

display(regression_results)

best_regression_model = rf_regressor if rf_r2 >= linear_r2 else linear_regression
best_regression_name = "Random Forest Regressor" if rf_r2 >= linear_r2 else "Linear Regression"

print("Best Regression Model:", best_regression_name)


# PART 2 — Classification: Predict Visit Mode

In [ ]:
if "VisitMode" not in df.columns:
    raise ValueError("VisitMode column was not found.")

classification_features = [c for c in candidate_features if c in df.columns]

X_class = df[classification_features].copy()
y_class = df["VisitMode"].copy()

# Remove classes with fewer than 2 samples for stratified splitting
class_counts = y_class.value_counts()
valid_classes = class_counts[class_counts >= 2].index
mask = y_class.isin(valid_classes)

X_class = X_class[mask]
y_class = y_class[mask]

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_class
)

categorical_features_class = X_class.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features_class = [
    c for c in X_class.columns
    if c not in categorical_features_class
]

preprocessor_class = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features_class),
        ("numerical", "passthrough", numerical_features_class)
    ]
)


In [ ]:
# Logistic Regression
logistic_classifier = Pipeline([
    ("preprocessor", preprocessor_class),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_classifier.fit(X_train_class, y_train_class)
log_predictions = logistic_classifier.predict(X_test_class)

log_accuracy = accuracy_score(y_test_class, log_predictions)
log_f1 = f1_score(y_test_class, log_predictions, average="weighted", zero_division=0)

print("Logistic Regression")
print("Accuracy:", log_accuracy)
print("F1 Score:", log_f1)


In [ ]:
# Random Forest Classifier
rf_classifier = Pipeline([
    ("preprocessor", preprocessor_class),
    ("model", RandomForestClassifier(
        n_estimators=150,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_classifier.fit(X_train_class, y_train_class)
rf_class_predictions = rf_classifier.predict(X_test_class)

rf_accuracy = accuracy_score(y_test_class, rf_class_predictions)
rf_precision = precision_score(y_test_class, rf_class_predictions, average="weighted", zero_division=0)
rf_recall = recall_score(y_test_class, rf_class_predictions, average="weighted", zero_division=0)
rf_f1 = f1_score(y_test_class, rf_class_predictions, average="weighted", zero_division=0)

print("Random Forest Classifier")
print("Accuracy:", rf_accuracy)
print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("F1 Score:", rf_f1)

print("\nClassification Report:")
print(classification_report(y_test_class, rf_class_predictions, zero_division=0))


In [ ]:
classification_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest Classifier"],
    "Accuracy": [log_accuracy, rf_accuracy],
    "F1 Score": [log_f1, rf_f1]
})

display(classification_results)

best_classification_model = (
    rf_classifier if rf_f1 >= log_f1 else logistic_classifier
)

best_classification_name = (
    "Random Forest Classifier"
    if rf_f1 >= log_f1
    else "Logistic Regression"
)

print("Best Classification Model:", best_classification_name)


# PART 3 — Recommendation System

In [ ]:
required_cols = ["UserId", "AttractionId", "Rating"]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Recommendation system requires these columns: {missing}")

attraction_name_col = "Attraction" if "Attraction" in df.columns else "AttractionId"

user_item_matrix = df.pivot_table(
    index="UserId",
    columns=attraction_name_col,
    values="Rating",
    aggfunc="mean"
)

print("User-Attraction Matrix Shape:", user_item_matrix.shape)
display(user_item_matrix.head())


In [ ]:
def recommend_attractions(user_id, n=5):
    if user_id not in user_item_matrix.index:
        return pd.DataFrame({
            "Message": ["User ID not found in the dataset."]
        })

    user_ratings = user_item_matrix.loc[user_id]
    visited_attractions = user_ratings.dropna().index.tolist()

    recommendations = user_item_matrix.mean(axis=0).sort_values(
        ascending=False
    )

    recommendations = recommendations[
        ~recommendations.index.isin(visited_attractions)
    ].head(n)

    return pd.DataFrame({
        "Recommended Attraction": recommendations.index,
        "Predicted Preference Score": recommendations.values
    })

example_user = user_item_matrix.index[0]
print("Example User:", example_user)

recommend_attractions(example_user, n=5)


# Save Models and Processed Data

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

joblib.dump(
    best_regression_model,
    "models/best_regression_model.joblib"
)

joblib.dump(
    best_classification_model,
    "models/best_classification_model.joblib"
)

joblib.dump(
    user_item_matrix,
    "models/user_item_matrix.joblib"
)

df.to_csv(
    "data/cleaned_tourism_dataset.csv",
    index=False
)

print("Files saved successfully:")
for root, dirs, files_ in os.walk("."):
    if root in ["./models", "./data"]:
        for file in files_:
            print(os.path.join(root, file))


# Download Project Files

In [ ]:
from google.colab import files

files.download("models/best_regression_model.joblib")
files.download("models/best_classification_model.joblib")
files.download("models/user_item_matrix.joblib")
files.download("data/cleaned_tourism_dataset.csv")


# Business Insights

1. Identify popular attractions and attraction types using visit counts.
2. Analyze rating trends to identify highly rated and low-rated attractions.
3. Predict visit modes to support personalized tourism marketing.
4. Recommend attractions based on historical user ratings.
5. Use geographical and demographic patterns to understand tourism trends.

# Project Deliverables

- Cleaned tourism dataset
- Exploratory data analysis and visualizations
- Regression model for rating prediction
- Classification model for visit mode prediction
- Recommendation system
- Saved trained models
- Streamlit deployment-ready project
